In [0]:
from pyspark.sql.functions import xxhash64, col, avg, count, round as _round

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# dim_movies
# xxhash64 escolhido em vez de monotonically_increasing_id porque e deterministico entre execucoes
df_dim_movies = spark.read.table("silver.tb_info_filmes") \
    .withColumn("sk_movie_id", xxhash64(col("id_filme"))) \
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento",
            "ano_lancamento", "duracao_minutos", "idioma_original",
            "status_filme", "sinopse")
df_dim_movies.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_movies")

# dim_genres
df_dim_genres = spark.read.table("silver.tb_generos").select("genero").distinct() \
    .withColumn("sk_genre_id", xxhash64(col("genero"))) \
    .withColumnRenamed("genero", "nome_genero")
df_dim_genres.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_genres")

# dim_people
# hash de nome mais tipo porque a mesma pessoa pode ocupar papeis distintos, como diretor e roteirista
df_dim_people = spark.read.table("silver.tb_pessoas_empresas") \
    .filter("tipo_entidade IN ('Ator', 'Diretor', 'Roteirista')") \
    .select("nome_entidade", "tipo_entidade").distinct() \
    .withColumn("sk_person_id", xxhash64(col("nome_entidade"), col("tipo_entidade"))) \
    .withColumnRenamed("nome_entidade", "nome_pessoa") \
    .withColumnRenamed("tipo_entidade", "tipo_pessoa") \
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
df_dim_people.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_people")

# dim_companies
df_dim_companies = spark.read.table("silver.tb_pessoas_empresas") \
    .filter("tipo_entidade = 'Produtora'") \
    .select("nome_entidade").distinct() \
    .withColumn("sk_company_id", xxhash64(col("nome_entidade"))) \
    .withColumnRenamed("nome_entidade", "nome_produtora") \
    .select("sk_company_id", "nome_produtora")
df_dim_companies.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_companies")

# dim_reviews
# spec pede uma linha por filme com contagem e media, e cast double porque avg sobre decimal devolve decimal
df_dim_reviews = spark.read.table("silver.tb_avaliacoes_usuarios") \
    .groupBy("id_filme") \
    .agg(
        count("nota_usuario").alias("qtd_avaliacoes_usuarios"),
        _round(avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    ) \
    .withColumn("sk_movie_id", xxhash64(col("id_filme"))) \
    .withColumn("sk_review_id", xxhash64(col("id_filme"), col("qtd_avaliacoes_usuarios"))) \
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
df_dim_reviews.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_reviews")

# bridge_movie_genre
df_bridge_genre = spark.read.table("silver.tb_generos").alias("sg") \
    .join(spark.read.table("gold.dim_movies").alias("m"),
          col("sg.id_filme") == col("m.id_filme"), "inner") \
    .join(spark.read.table("gold.dim_genres").alias("g"),
          col("sg.genero") == col("g.nome_genero"), "inner") \
    .select(col("m.sk_movie_id"), col("g.sk_genre_id")).distinct()
df_bridge_genre.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.bridge_movie_genre")

# bridge_movie_person
df_bridge_person = spark.read.table("silver.tb_pessoas_empresas").alias("sp") \
    .filter("tipo_entidade IN ('Ator', 'Diretor', 'Roteirista')") \
    .join(spark.read.table("gold.dim_movies").alias("m"),
          col("sp.id_filme") == col("m.id_filme"), "inner") \
    .join(spark.read.table("gold.dim_people").alias("p"),
          (col("sp.nome_entidade") == col("p.nome_pessoa")) &
          (col("sp.tipo_entidade") == col("p.tipo_pessoa")), "inner") \
    .select(col("m.sk_movie_id"), col("p.sk_person_id")).distinct()
df_bridge_person.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.bridge_movie_person")

# bridge_movie_company
df_bridge_company = spark.read.table("silver.tb_pessoas_empresas").alias("sp") \
    .filter("tipo_entidade = 'Produtora'") \
    .join(spark.read.table("gold.dim_movies").alias("m"),
          col("sp.id_filme") == col("m.id_filme"), "inner") \
    .join(spark.read.table("gold.dim_companies").alias("c"),
          col("sp.nome_entidade") == col("c.nome_produtora"), "inner") \
    .select(col("m.sk_movie_id"), col("c.sk_company_id")).distinct()
df_bridge_company.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.bridge_movie_company")

# fact_movies_performance
# enunciado restringe a fato a filmes lancados, e as notas levam cast double por exigencia de schema da spec
df_fact = spark.read.table("gold.dim_movies").alias("m") \
    .filter(col("status_filme") == "Lançado") \
    .join(spark.read.table("silver.tb_financeiro_filmes").alias("f"),
          col("m.id_filme") == col("f.id_filme"), "left") \
    .join(spark.read.table("silver.tb_metricas_engajamento").alias("e"),
          col("m.id_filme") == col("e.id_filme"), "left") \
    .select(
        col("m.sk_movie_id"),
        col("f.orcamento_usd"), col("f.receita_usd"), col("f.lucro_usd"),
        col("f.orcamento_brl"), col("f.receita_brl"), col("f.lucro_brl"),
        col("e.popularidade").cast("double"),
        col("e.nota_media_tmdb").cast("double"),
        col("e.qtd_votos_tmdb"),
        col("e.nota_media_imbd").cast("double"),
        col("e.qtd_votos_imbd")
    )
df_fact.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.fact_movies_performance")